# Sales Forecasting with PyCaret
# Part B: Modelling
## Dr José M Albornoz
### February 2025

This notebook presents an example of how Pycaret can be used for time series forecasting. The dataset used in this example describe retail sales at a department store; no exogenous variables are considered.

# 0.- Imports

In [1]:
from pycaret.time_series import *
import pandas as pd
import numpy as np
import plotly.express as px
import datetime

# maximum number of dataframe rows and columns displayed
pd.set_option('display.max_rows', 50000)
pd.set_option('display.max_columns', 500)

pd.options.mode.chained_assignment = None

RANDOM_SEED = 801

# 1.- Load data

In [2]:
data = pd.read_csv('../data/sales_without_exogenous.csv')
data['Date'] = pd.to_datetime(data['Date'])
data.set_index('Date', inplace=True)
target = "Sales"
data.head()

FileNotFoundError: [Errno 2] No such file or directory: '../data/sales_without_exogenous.csv'

In [ ]:
data.shape

In [ ]:
data.dtypes

In [ ]:
type(data.index)

# 2.- PyCaret Setup

The `setup` function initializes the training environment and creates the transformation pipeline. Setup function must be called before executing any other function in PyCaret. Setup has only one required parameter i.e. `data`. All the other parameters are optional.

In [ ]:
# We want to forecast the next 14 days, and we will use 3 fold cross-validation to test the models.
fh = 14 # or alternately fh = np.arange(1,13)
fold = 3 # (default)

In [ ]:
s = setup(data, fh = 14, session_id = RANDOM_SEED)

# 3.- Available models

The following function lists all the time series models available for training.

In [ ]:
# check available models
models()

The `compare_models` function trains and evaluates the performance of all the estimators available in the model library using cross-validation. The output of this function is a scoring grid with average cross-validated scores. Metrics evaluated during CV can be accessed using the get_metrics function. Custom metrics can be added or removed using add_metric and remove_metric function.

You can use the `include` and `exclude` parameter in the `compare_models` to train only selected models or exclude specific models from training by passing the model id's in `exclude` parameter.

In [ ]:
#best = compare_models(sort='MASE', include = ['naive', 'snaive', 'arima', 'exp_smooth', 'xgboost_cds_dt', 'dt_cds_dt'])

In [ ]:
best = compare_models(sort='MASE', include = ['naive', 'snaive', 'arima', 'exp_smooth', 'xgboost_cds_dt', 'dt_cds_dt', 'bats', 'tbats'])

In [ ]:
#best = compare_models(sort='MASE', include = ['naive', 'snaive', 'arima', 'exp_smooth', 'xgboost_cds_dt', 'dt_cds_dt', 'grand_means',
#                                              'polytrend', 'ets', 'theta', 'stlf', 'croston', 'bats', 'tbats'])

In [ ]:
best

The function above has return the trained model object as an output. If you need access to the scoring grid you can use `pull` function to access the dataframe.

The score on the test set can be accessed using predict_model on the best performing model:

In [ ]:
prediction_holdout = predict_model(best)

# 4.- Analyse model

A first step in analysing the performance of the BATS model is to examine the residuals. For a well-fitting time series model the residuals should resemble white noise, meaning that the model has captured all the systematic information in the data, and only random fluctuations remain. In the ideal case, the residuals should be normally distributed, with a mean close to zero and without any significant autocorrelation.

In [ ]:
# check_stats on residuals of best model
check_stats(estimator = best)

You can use the plot_model function to analyzes the performance of a trained model on the test set. It may require re-training the model in certain cases.

In [ ]:
# residuals plot
plot_model(best, plot = 'residuals', fig_kwargs={"height": 400, "width": 1100})

To further assess the fit provided by the model, we will plot the forecast for the specified 14 days horizon against the actual data: 

In [ ]:
# plot forecast
plot_model(best, plot = 'forecast', fig_kwargs={"height": 400, "width": 1100})

In [ ]:
# plot forecast for the next 35 days
plot_model(best, plot = 'forecast', data_kwargs = {'fh' : 35})

# 5.- Forecasting

As seen in the previous section, 

In [ ]:
# predict on test set
holdout_pred = predict_model(best)

In [ ]:
# show predictions df
holdout_pred.head()

In [ ]:
# generate forecast for 36 period in future
predict_model(best, fh = 36)

# 6.- Save model

In [ ]:
# save pipeline
save_model(best, 'time_series_v0')

In [ ]:
# load pipeline
loaded_best_pipeline = load_model('time_series_v0')
loaded_best_pipeline

# 7.- Experiment Logging

PyCaret integrates with many different type of experiment loggers (default = 'mlflow'). To turn on experiment tracking in PyCaret you can set `log_experiment` and `experiment_name` parameter. It will automatically track all the metrics, hyperparameters, and artifacts based on the defined logger.

In [ ]:
# s = setup(data, fh = 3, session_id = 123, log_experiment='mlflow', experiment_name='airline_experiment')

In [ ]:
# compare models
# best = compare_models()

In [ ]:
# start mlflow server on localhost:5000
# !mlflow ui

By default PyCaret uses MLFlow logger; that can be changed using `log_experiment` parameter. Following loggers are available:

- mlflow
- wandb
- comet_ml
- dagshub
  
For more information check out the docstring of the setup function.

In [ ]:
help(setup)

# 8.- Create model

This function trains and evaluates the performance of a given estimator using cross-validation. The output of this function is a scoring grid with CV scores by fold. Metrics evaluated during CV can be accessed using the `get_metrics` function. Custom metrics can be added or removed using `add_metric` and `remove_metric` function. All the available models can be accessed using the `models` function.

In [ ]:
# train ets with default fold=3
ets = create_model('ets')

The function above has return trained model object as an output. The scoring grid is only displayed and not returned. If you need access to the scoring grid you can use `pull` function to access the dataframe.

In [ ]:
ets_results = pull()
print(type(ets_results))
ets_results

In [ ]:
# train theta model with fold=5
theta = create_model('theta', fold=5)

In [ ]:
# train theta with specific model parameters
create_model('theta', deseasonalize = False, fold=5)

Some other parameters that you might find very useful in create_model are:

- cross_validation
- engine
- fit_kwargs
  
You can check the docstring of the function for more info.

# 9.- Tune model

The `tune_model` function tunes the hyperparameters of the model. The output of this function is a scoring grid with cross-validated scores by fold. The best model is selected based on the metric defined in `optimize` parameter. Metrics evaluated during cross-validation can be accessed using the `get_metrics` function. Custom metrics can be added or removed using `add_metric` and `remove_metric` function.

Metric to optimize can be defined in `optimize` parameter (default = 'MASE'). Also, a custom tuned grid can be passed with `custom_grid` parameter.

In [ ]:
# train a dt model with default params
dt = create_model('dt_cds_dt')

In [ ]:
# tune hyperparameters of best model 
tuned_dt = tune_model(dt)

In [ ]:
# define tuning grid
dt_grid = {'regressor__max_depth' : [None, 2, 4, 6, 8, 10, 12]}

# tune model with custom grid and metric = MAE
tuned_dt = tune_model(dt, custom_grid = dt_grid, optimize = 'MAE')

In [ ]:
# see tuned_best params
tuned_dt

In [ ]:
# to access the tuner object you can set return_tuner = True
tuned_best, tuner = tune_model(dt, return_tuner=True)

In [ ]:
# model object
tuned_best

In [ ]:
# tuner object
tuner

For more details on all available search_library and search_algorithm please check the docstring. Some other parameters that you might find very useful in tune_model are:

- choose_better
- custom_scorer
- n_iter
- search_algorithm
- optimize
- 
You can check the docstring of the function for more info.

In [ ]:
help(tune_model)

# 10.- Blend models

This function trains a `EnsembleForecaster` for select models passed in the `estimator_list` parameter. The output of this function is a scoring grid with CV scores by fold. Metrics evaluated during CV can be accessed using the `get_metrics` function. Custom metrics can be added or removed using `add_metric` and `remove_metric` function.

In [ ]:
# blend top 3 models
blend_models([best, tuned_dt])

Some other parameters that you might find very useful in blend_models are:

- choose_better
- method
- weights
- fit_kwargs
- optimize
- 
You can check the docstring of the function for more info.

In [ ]:
help(blend_models)

# 11.- Plot model

In [ ]:
# plot acf
# for certain plots you don't need a trained model
plot_model(plot = 'acf')

In [ ]:
# plot diagnostics
# for certain plots you don't need a trained model
plot_model(plot = 'diagnostics', fig_kwargs={'height': 600, "width": 800})

Some other parameters that you might find very useful in plot_model are:

- fig_kwargs
- data_kwargs
- display_format
- return_fig
- return_data
- save

You can check the docstring of the function for more info.

In [ ]:
help(plot_model)

# 12.- Finalise model

This function trains a given model on the entire dataset including the hold-out set.

In [ ]:
final_best = finalize_model(best)

In [ ]:
final_best

# 13.- Deploy model

This function deploys the entire ML pipeline on the cloud.

AWS: When deploying model on AWS S3, environment variables must be configured using the command-line interface. To configure AWS environment variables, type aws configure in terminal. The following information is required which can be generated using the Identity and Access Management (IAM) portal of your amazon console account:

- AWS Access Key ID
- AWS Secret Key Access
- Default Region Name (can be seen under Global settings on your AWS console)
- Default output format (must be left blank)

GCP: To deploy a model on Google Cloud Platform ('gcp'), the project must be created using the command-line or GCP console. Once the project is created, you must create a service account and download the service account key as a JSON file to set environment variables in your local environment. Learn more about it: https://cloud.google.com/docs/authentication/production

Azure: To deploy a model on Microsoft Azure ('azure'), environment variables for the connection string must be set in your local environment. Go to settings of storage account on Azure portal to access the connection string required. AZURE_STORAGE_CONNECTION_STRING (required as environment variable) Learn more about it: https://docs.microsoft.com/en-us/azure/storage/blobs/storage-quickstart-blobs-python?toc=%2Fpython%2Fazure%2FTOC.json

In [ ]:
# deploy model on aws s3
# deploy_model(best, model_name = 'my_first_platform_on_aws',
#             platform = 'aws', authentication = {'bucket' : 'pycaret-test'})

In [ ]:
# load model from aws s3
# loaded_from_aws = load_model(model_name = 'my_first_platform_on_aws', platform = 'aws',
#                              authentication = {'bucket' : 'pycaret-test'})

# loaded_from_aws

# 14.- Save/load experiment

This function saves all the experiment variables on disk, allowing to later resume without rerunning the setup function.

In [ ]:
# save experiment
save_experiment('my_experiment')

In [ ]:
# load experiment from disk
exp_from_disk = load_experiment('my_experiment', data=data)